# BigQuery: Global Queries Demo (February 2026 Preview)

This notebook demonstrates the **Global Queries** feature in BigQuery, which allows you to reference data stored in more than one region in a single query without needing to physically move or copy the data.

## Use Case
A global enterprise has sales data in `us-central1` and inventory data in `europe-west1`. Previously, they had to export/import data to join these tables. Now, they can join them directly.

### Requirements
- BigQuery API enabled.
- Datasets in at least two different regions (e.g., `us-central1` and `europe-west1`).
- Appropriate IAM permissions to access both datasets.

In [ ]:
# 1. Setup and Authentication
from google.colab import auth
auth.authenticate_user()
print('Authenticated')

from google.cloud import bigquery
project_id = 'YOUR_PROJECT_ID'  # @param {type:"string"}
client = bigquery.Client(project=project_id)

### 2. Define the Global Query

In this example, we join a US-based sales table with a European inventory table.

In [ ]:
query = """
SELECT
    sales.product_id,
    sales.units_sold,
    inventory.stock_level,
    inventory.warehouse_location
FROM
    `{project_id}.us_sales.sales_data` AS sales
INNER JOIN
    `{project_id}.eu_inventory.inventory_data` AS inventory
ON
    sales.product_id = inventory.product_id
WHERE
    sales.sale_date = CURRENT_DATE()
""".format(project_id=project_id)

print("Executing Global Query...")
query_job = client.query(query)
results = query_job.to_dataframe()
results.head()

### 3. Things to remember or know
- **Zero Data Movement**: No more costly and slow ETL pipelines to sync data across regions just for reporting.
- **Simpler Governance**: Maintain data residency compliance by keeping data in its home region while still allowing global insights.
- **Reduced Latency**: Insights are generated directly by BigQuery's global execution engine.